<a href="https://colab.research.google.com/github/chshft1018-hub/TCG_management/blob/main/%E5%8D%A1%E7%89%8C%E6%8A%95%E8%B3%87%E7%AE%A1%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 環境建置

In [30]:
!pip install playwright nest_asyncio pandas
!playwright install chromium
!playwright install-deps

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 128 kB in 1s (129 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is alre

#　爬蟲邏輯

In [45]:
import asyncio

import nest_asyncio

import pandas as pd

import aiohttp

import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

from datetime import datetime, timedelta



nest_asyncio.apply()



# --- 1. 資料獲取層 ---

async def get_chart_data(product_id, option_id):

    # salesChartOptionId: -1=全部, 18=A品, 22=PSA10

    url = f"https://snkrdunk.com/v1/apparels/{product_id}/sales-chart/used?range=all&salesChartOptionId={option_id}"

    headers = {"User-Agent": "Mozilla/5.0"}

    async with aiohttp.ClientSession() as session:

        async with session.get(url, headers=headers) as resp:

            return await resp.json() if resp.status == 200 else None



# --- 2. 計算與邏輯層 ---

def analyze_data(json_data, cost_twd, rate):

    if not json_data or 'points' not in json_data:

        return {"latest": 0, "avg_1w": 0, "avg_1m": 0, "avg_3m": 0, "roi": 0, "count": 0}



    df = pd.DataFrame(json_data['points'], columns=['timestamp', 'price_jpy'])

    df['date'] = pd.to_datetime(df['timestamp'], unit='ms')

    df['price_twd'] = df['price_jpy'] * rate



    now = pd.Timestamp.now()

    latest = df.iloc[-1]['price_twd']



    # 計算均價

    avg_1w = df[df['date'] >= (now - pd.Timedelta(days=7))]['price_twd'].mean()

    avg_1m = df[df['date'] >= (now - pd.Timedelta(days=30))]['price_twd'].mean()

    avg_3m = df[df['date'] >= (now - pd.Timedelta(days=90))]['price_twd'].mean()



    roi = ((latest - cost_twd) / cost_twd * 100)



    return {

        "latest": latest, "avg_1w": avg_1w, "avg_1m": avg_1m, "avg_3m": avg_3m,

        "roi": roi, "count": len(df)

    }



# --- 3. UI 顯示層 ---

def show_dashboard(metrics_A, metrics_PSA, product_id, cost):

    def card(title, m):

        roi_color = "#4ade80" if m['roi'] >= 0 else "#FF0000"

        return f"""

        <div style="background:#2a2a2a; border-radius:10px; padding:15px; width:45%; color:white;">

            <h3>{title}</h3>

            <p>最新: NT$ {m['latest']:,.0f}</p>

            <p>1週均: NT$ {m['avg_1w']:,.0f} | 1月均: NT$ {m['avg_1m']:,.0f} | 3月均: NT$ {m['avg_3m']:,.0f}</p>

            <div style="color:{roi_color}; font-size:20px; font-weight:bold;">ROI: {m['roi']:.2f}%</div>

        </div>"""



    html = f"<div style='display:flex; gap:20px;'>{card('裸卡(A品)', metrics_A)}{card('鑑定卡(PSA10)', metrics_PSA)}</div>"

    display(HTML(html))



# --- 4. 系統入口 ---

ui_id = widgets.Text(value='826553', description='商品 ID:')

ui_cost = widgets.FloatText(value=15000, description='成本(NT$):')

ui_button = widgets.Button(description='立即分析', button_style='success')

ui_output = widgets.Output()



def run_system(b):

    with ui_output:

        clear_output()

        print("正在透過 API 撈取數據...")

        # 分別抓取 A品 (id:18) 與 PSA10 (id:22)

        data_A = asyncio.run(get_chart_data(ui_id.value, 18))

        data_PSA = asyncio.run(get_chart_data(ui_id.value, 22))



        metrics_A = analyze_data(data_A, ui_cost.value, 0.20)

        metrics_PSA = analyze_data(data_PSA, ui_cost.value, 0.20)



        show_dashboard(metrics_A, metrics_PSA, ui_id.value, ui_cost.value)

async def get_product_name(product_id):

# 這是商品資訊的 API 端點

    url = f"https://snkrdunk.com/v1/apparels/{product_id}"

    headers = {"User-Agent": "Mozilla/5.0"}

    async with aiohttp.ClientSession() as session:

        try:

            async with session.get(url, headers=headers) as resp:

                data = await resp.json()

                # 從 API 回傳的 JSON 中提取商品名稱

                return data.get('name', '未知卡牌')

        except:

            return '未知卡牌'



ui_button.on_click(run_system)

display(widgets.VBox([ui_id, ui_cost, ui_button, ui_output]))
# --- 重新定義正規化後的欄位 ---
columns = [
    '卡片編號', '卡片名稱', '卡片成本',
    'A品最新價', 'A品週均', 'A品月均', 'ROI(A品)',
    'PSA10最新價', 'PSA10週均', 'PSA10月均', 'ROI(PSA10)'
]

def run_system_refined(b):
    with ui_output:
        clear_output()
        print("正在透過 API 撈取數據...")

        # 1. 取得 API 資料
        id_val, cost_val = ui_id.value, ui_cost.value
        data_A = asyncio.run(get_chart_data(id_val, 18))
        data_PSA = asyncio.run(get_chart_data(id_val, 22))
        name = asyncio.run(get_product_name(id_val))

        m_A = analyze_data(data_A, cost_val, 0.20)
        m_PSA = analyze_data(data_PSA, cost_val, 0.20)

        # 2. 顯示 Dashboard
        show_dashboard(m_A, m_PSA, id_val, cost_val)

        # 3. 填入完整欄位資料
        global portfolio_db
        new_row = {
            '卡片編號': id_val,
            '卡片名稱': name,
            '卡片成本': cost_val,
            'A品最新價': f"{m_A['latest']:,.0f}",
            'A品週均': f"{m_A['avg_1w']:,.0f}",
            'A品月均': f"{m_A['avg_1m']:,.0f}",
            'ROI(A品)': f"{m_A['roi']:.2f}%",
            'PSA10最新價': f"{m_PSA['latest']:,.0f}",
            'PSA10週均': f"{m_PSA['avg_1w']:,.0f}",
            'PSA10月均': f"{m_PSA['avg_1m']:,.0f}",
            'ROI(PSA10)': f"{m_PSA['roi']:.2f}%"
        }

        # 4. 更新表格
        portfolio_db = pd.concat([portfolio_db, pd.DataFrame([new_row])], ignore_index=True)
        print("\n📊 即時庫存報表：")
        display(portfolio_db[columns])

# 確保按鈕綁定 (請確認這段程式碼在 UI 定義後執行)
ui_button.on_click(run_system_refined)
# --- 修正與刪除整合控制區 ---
# 1. 修正區塊
ui_edit_idx = widgets.IntText(value=0, description='要修改列號:')
ui_new_cost = widgets.FloatText(description='新成本:')
ui_edit_btn = widgets.Button(description='修正單筆成本', button_style='warning')

# 2. 刪除區塊
ui_start_idx = widgets.IntText(value=0, description='起始列號:')
ui_end_idx = widgets.IntText(value=0, description='結束列號:')
ui_range_del_btn = widgets.Button(description='刪除指定區間', button_style='danger')

def update_cost(b):
    global portfolio_db
    idx = ui_edit_idx.value
    if idx in portfolio_db.index:
        portfolio_db.at[idx, '卡片成本'] = ui_new_cost.value
        with ui_output:
            clear_output(wait=True)
            display(portfolio_db)

def range_delete(b):
    global portfolio_db
    start, end = ui_start_idx.value, ui_end_idx.value
    # 刪除區間並重置索引以維持表格整潔
    portfolio_db = portfolio_db.drop(portfolio_db.index[start : end + 1]).reset_index(drop=True)
    with ui_output:
        clear_output(wait=True)
        display(portfolio_db)

ui_edit_btn.on_click(update_cost)
ui_range_del_btn.on_click(range_delete)

# UI 顯示整合
display(widgets.VBox([
    widgets.HBox([ui_edit_idx, ui_new_cost, ui_edit_btn]),
    widgets.HBox([ui_start_idx, ui_end_idx, ui_range_del_btn])
]))

# Google sheet

In [32]:
!pip install gspread oauth2client

In [48]:
from google.colab import auth
import gspread
from google.auth import default

# 1. 執行授權
auth.authenticate_user()
creds, _ = default()

# 2. 建立連線 (這是目前 gspread 推薦的正確做法)
gc = gspread.authorize(creds)

def export_to_google_sheet(b):
    try:
        # 1. 處理資料：將所有 NaN 或無限大值替換為空字串或 0
        df_to_export = portfolio_db.fillna('')

        # 2. 開啟表單
        sh = gc.open('卡牌管理')
        worksheet = sh.sheet1

        # 3. 清空並寫入 (使用處理過後的資料)
        worksheet.clear()
        data = [df_to_export.columns.values.tolist()] + df_to_export.values.tolist()
        worksheet.update(data)

        print("✅ 已成功同步至 Google Sheet!")
    except Exception as e:
        print(f"❌ 同步失敗：{e}")

# 3. 按鈕綁定
ui_sheet_btn = widgets.Button(description='同步至 Google Sheet', button_style='primary')
ui_sheet_btn.on_click(export_to_google_sheet)
display(ui_sheet_btn)
def export_to_google_sheet(b):
    try:
        # 開啟表單並寫入
        sh = gc.open('卡牌管理')
        worksheet = sh.sheet1

        # 清空表單以消除舊欄位遺留
        worksheet.clear()

        # 寫入正確資料
        data = [df_to_export.columns.values.tolist()] + df_to_export.values.tolist()
        worksheet.update(data)

        print("✅ 報表已對齊並成功同步至 Google Sheet!")
    except Exception as e:
        print(f"❌ 同步失敗，請確認欄位名稱是否吻合：{e}")

Button(button_style='primary', description='同步至 Google Sheet', style=ButtonStyle())